## 라이브러리 호출 및 데이터 로드

In [1]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import platform
import ast
from collections import Counter
import json
from pprint import pprint
import warnings
import platform

# 통계용 라이브러리 호출
from scipy import stats
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from itertools import combinations
from matplotlib.patches import Patch
import pingouin as pg
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize, proportion_confint
from statsmodels.stats.power import NormalIndPower

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

In [31]:
# 데이터 로드
df_champion = pd.read_csv('../../유저단위_게임데이터_상위랭커보존-stats_champion_1.csv')
df_combination = pd.read_csv('../../유저단위_게임데이터_상위랭커보존-stats_combination_1.csv')
df_champion_with_items = pd.read_csv("../../유저단위_게임데이터_상위랭커보존-stats_champion_items_1.csv")

---
#### **➡️ 경기데이터와 챔피언 정보가 담긴 테이블 제작**
- `df_champion`

In [32]:
df_champion.info()

<class 'pandas.DataFrame'>
RangeIndex: 396204 entries, 0 to 396203
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   user_id           396204 non-null  str  
 1   game_id           396204 non-null  str  
 2   user_tier         396204 non-null  str  
 3   ranked            396204 non-null  int64
 4   flag_1            396204 non-null  int64
 5   flag_2            396204 non-null  int64
 6   active_synergies  396204 non-null  str  
 7   top4_flag         396204 non-null  bool 
 8   ranked_1          396204 non-null  bool 
 9   champions         396204 non-null  str  
dtypes: bool(2), int64(3), str(5)
memory usage: 24.9 MB


In [33]:
df_champion.head(1)

,user_id,game_id,user_tier,ranked,flag_1,flag_2,active_synergies,top4_flag,ranked_1,champions
0,KR-USER-1,KR_4291707834,platinum,5,0,0,{},False,False,"[{'name': 'ziggs', 'star': 1, 'cost': 1, 'origin': 'Rebel', 'class': ""['Demolitionist']""}, {'name': 'ashe', 'star': 1, 'cost': 3, 'origin': 'Celestial', 'class': ""['Sniper']""}, {'name': 'chogath', 'star': 1, 'cost': 4, 'origin': 'Void', 'class': ""['Brawler']""}, {'name': 'ekko', 'star': 1, 'cost': 5, 'origin': 'Cybernetic', 'class': ""['Infiltrator']""}]"


#### 🔻분석단위를 챔피언 단위로 변경
- 하나의 row = 한 유저가 참여한 하나의 게임
- 챔피언마다 name, star, cost, origin, class 정보가 담겨있음 

In [34]:
# champions 컬럼 문자열 → 리스트로 변환
df_champion['champions'] = df_champion['champions'].apply(ast.literal_eval)

# 챔피언 단위로 분리 (explode)
df_champ_exploded = df_champion.explode('champions')

# 딕셔너리 컬럼 분리
df_champ_exploded = pd.concat([
    df_champ_exploded.drop(columns='champions'),
    df_champ_exploded['champions'].apply(pd.Series)
], axis=1)
# champion 컬럼 분리 전
# champions = {'name': 'ziggs', 'star': 1, 'cost': 1, 'origin': 'Rebel', 'class': ['Demolitionist']}

# champion 컬럼 분리 후
# user_id    game_id  ranked  top4_flag   name   star  cost  origin         class
# KR-USER-1  KR_...     5       False     ziggs   1     1    Rebel     [Demolitionist]
# KR-USER-1  KR_...     5       False     ashe    1     3   Celestial     [Sniper]

print(df_champ_exploded.shape)
display(df_champ_exploded.head(1))

(3130701, 14)


,user_id,game_id,user_tier,ranked,flag_1,flag_2,active_synergies,top4_flag,ranked_1,name,star,cost,origin,class
0,KR-USER-1,KR_4291707834,platinum,5,0,0,{},False,False,ziggs,1,1,Rebel,['Demolitionist']


In [35]:
# flag_1 제거 (챔피언 정보 누락된 row 제거)
df_champ_exploded = df_champ_exploded[df_champ_exploded['flag_1'] == 0]

print(df_champ_exploded.shape)

(3130701, 14)


In [ ]:
# class 컬럼 확인
print(df_champ_exploded['class'].dtype)
display(df_champ_exploded['class'].head(1))

In [ ]:
# 문자열 → 리스트 변환
df_champ_exploded['class'] = df_champ_exploded['class'].apply(ast.literal_eval)

# 시너지 단위로 분리
df_synergy_exploded = df_champ_exploded.explode('class')

# (4) 전체 유저가 가장 많이 선택한 시너지 Top 5
print('[전체 유저가 가장 많이 선택한 시너지 Top 5]')
display(df_synergy_exploded['class'].value_counts().head(5))
print()

# (5) 티어별로 유저가 가장 많이 선택한 시너지 Top 5
print('[티어별로 유저가 가장 많이 선택한 시너지 Top 5]')
top5_synergy_by_tier = (
    df_synergy_exploded
    .groupby('user_tier')['class']
    .value_counts()
    .groupby('user_tier')
    .head(5)
    .reset_index()
)
display(top5_synergy_by_tier)

---
#### **➡️ 콤비네이션 정보가 담긴 테이블 제작**
- `df_combination`

In [43]:
print(df_combination.info())
display(df_combination.head(1))

<class 'pandas.DataFrame'>
RangeIndex: 396239 entries, 0 to 396238
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   gameid               396239 non-null  str  
 1   user_tier            396239 non-null  str  
 2   ranked               396239 non-null  int64
 3   user_id              396239 non-null  str  
 4   flag_1               396239 non-null  int64
 5   flag_2               396239 non-null  int64
 6   active_synergies     396239 non-null  str  
 7   top4_flag            396239 non-null  bool 
 8   ranked_1             396239 non-null  bool 
 9   combination_rebuild  396239 non-null  str  
dtypes: bool(2), int64(3), str(5)
memory usage: 24.9 MB
None


,gameid,user_tier,ranked,user_id,flag_1,flag_2,active_synergies,top4_flag,ranked_1,combination_rebuild
0,KR_4291707834,platinum,5,KR-USER-1,0,0,{},False,False,"{'Cybernetic': 1, 'Demolitionist': 1, 'Infiltrator': 1, 'Rebel': 1, 'Brawler': 1, 'Celestial': 1, 'Void': 1, 'Sniper': 1}"


In [ ]:
# (6) 전체 유저 기준에서 제작
# 문자열 → 딕셔너리 변환
df_combination['active_synergies'] = df_combination['active_synergies'].apply(ast.literal_eval)
df_combination['combination_rebuild'] = df_combination['combination_rebuild'].apply(ast.literal_eval)

# 활성화된 조합 리스트 추출
df_combination['combo_list_active'] = df_combination['active_synergies'].apply(
    lambda x: list(x.keys()) if x else []
)

# 보유한 조합 리스트 추출
df_combination['combo_list_rebuild'] = df_combination['combination_rebuild'].apply(
    lambda x: list(x.keys()) if x else []
)

# 가장 많이 활성화한 콤비네이션 Top 5
df_combo_active_exploded = df_combination.explode('combo_list_active')
print('[전체 유저가 가장 많이 활성화한 콤비네이션 Top 5]')
display(df_combo_active_exploded['combo_list_active'].value_counts().head(5))
print('='*50)

# 가장 많이 보유한 콤비네이션 Top 5
df_combo_rebuild_exploded = df_combination.explode('combo_list_rebuild')
print('[전체 유저가 가장 많이 보유한 콤비네이션 Top 5]')
display(df_combo_rebuild_exploded['combo_list_rebuild'].value_counts().head(5))

[전체 유저가 가장 많이 활성화한 콤비네이션 Top 5]


combo_list_active
Chrono        200384
Celestial     137136
Mercenary     127057
Blaster       115469
ManaReaver    113311
Name: count, dtype: int64

[전체 유저가 가장 많이 보유한 콤비네이션 Top 5]


combo_list_rebuild
Chrono        262897
Cybernetic    210292
Vanguard      191019
DarkStar      189045
Celestial     180392
Name: count, dtype: int64

In [45]:
# (7) 티어별 유저 기준에서 확인
# 활성화한 조합 Top 5
top5_active_by_tier = (
    df_combo_active_exploded
    .groupby('user_tier')['combo_list_active']
    .value_counts()
    .groupby('user_tier')
    .head(5)
    .reset_index()
)

display(top5_active_by_tier)

,user_tier,combo_list_active,count
0,challenger,Chrono,38273
1,challenger,Celestial,28024
2,challenger,Blaster,25234
3,challenger,Mercenary,24362
4,challenger,ManaReaver,23896
5,diamond,Chrono,44033
6,diamond,Mercenary,28890
7,diamond,Celestial,28364
8,diamond,ManaReaver,24393
9,diamond,Blademaster,22950


In [46]:
# 보유한 조합 Top 5
top5_rebuild_by_tier = (
    df_combo_rebuild_exploded
    .groupby('user_tier')['combo_list_rebuild']
    .value_counts()
    .groupby('user_tier')
    .head(5)
    .reset_index()
)

display(top5_rebuild_by_tier)

,user_tier,combo_list_rebuild,count
0,challenger,Chrono,49860
1,challenger,Cybernetic,42271
2,challenger,Vanguard,40567
3,challenger,Valkyrie,37310
4,challenger,Blaster,37012
5,diamond,Chrono,56554
6,diamond,Cybernetic,44308
7,diamond,DarkStar,38044
8,diamond,Valkyrie,37455
9,diamond,Blademaster,36878


---
---
### **➡️ 경기데이터와 챔피언 데이터, 아이템 데이터가 결합된 테이블**
- `df_champion_with_items`

In [49]:
print(df_champion_with_items.info())
display(df_champion_with_items.head(1))

<class 'pandas.DataFrame'>
RangeIndex: 396239 entries, 0 to 396238
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   gameid          396239 non-null  str  
 1   user_tier       396239 non-null  str  
 2   ranked          396239 non-null  int64
 3   champion        396239 non-null  str  
 4   item_count      396239 non-null  int64
 5   champion_count  396239 non-null  int64
dtypes: int64(3), str(3)
memory usage: 18.1 MB
None


,gameid,user_tier,ranked,champion,item_count,champion_count
0,KR_4291707834,platinum,5,"{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",4,4


In [51]:
# 문자열 → 딕셔너리 변환
df_champion_with_items['champion'] = df_champion_with_items['champion'].apply(ast.literal_eval)

# 챔피언별 아이템, 별 정보 리스트로 변환
df_champion_with_items['champion'] = df_champion_with_items['champion'].apply(
    lambda x: [{'name': k, 'items': v['items'], 'star': v['star']} for k, v in x.items()]
)

In [52]:
# 챔피언 단위로 explode
df_champ_item_exploded = df_champion_with_items.explode('champion')

# 딕셔너리 컬럼 분리 -> 최종 데이터프레임
df_champ_item = pd.concat([
    df_champ_item_exploded.drop(columns='champion'),
    df_champ_item_exploded['champion'].apply(pd.Series)
], axis=1)

print(df_champ_item.shape)
display(df_champ_item.head(1))

(3130736, 9)


,gameid,user_tier,ranked,item_count,champion_count,name,items,star,0
0,KR_4291707834,platinum,5,4,4,Ziggs,[7],1.0,NaN
0,KR_4291707834,platinum,5,4,4,Ashe,[9],1.0,NaN


In [ ]:
# 0 컬럼이 생기면 제거
if 0 in df_champ_item.columns:
    df_champ_item = df_champ_item.drop(columns=[0])

print(type(df_champ_item['items'].iloc[0])) # list
print(df_champ_item['items'].iloc[0])

<class 'list'>
[7]


In [62]:
df_champ_item.info()

<class 'pandas.DataFrame'>
Index: 3130736 entries, 0 to 396238
Data columns (total 8 columns):
 #   Column          Dtype  
---  ------          -----  
 0   gameid          str    
 1   user_tier       str    
 2   ranked          int64  
 3   item_count      int64  
 4   champion_count  int64  
 5   name            str    
 6   items           object 
 7   star            float64
dtypes: float64(1), int64(3), object(1), str(3)
memory usage: 215.0+ MB


In [63]:
# 아이템 테이블 로드
df_item = pd.read_csv('../../tft_game_dataset/TFT_Item_CurrentVersion.csv')

In [64]:
# 아이템 테이블 확인
print(df_item.info())
display(df_item.head(3))

<class 'pandas.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      54 non-null     int64
 1   name    54 non-null     str  
dtypes: int64(1), str(1)
memory usage: 996.0 bytes
None


,id,name
0,1,B.F. Sword
1,2,Recurve Bow
2,3,Needlessly Large Rod


In [66]:
# 아이템 ID + 이름 매핑 딕셔너리
item_map = dict(zip(df_item['id'], df_item['name']))

# items 컬럼의 ID를 이름으로 변환 -> 새로운 컬럼을 만들어서 진행
df_champ_item['item_names'] = df_champ_item['items'].apply(
    lambda x: [item_map.get(i, 'Unknown') for i in x] if isinstance(x, list) else []
)

In [67]:
display(df_champ_item.head(4))

,gameid,user_tier,ranked,item_count,champion_count,name,items,star,item_names
0,KR_4291707834,platinum,5,4,4,Ziggs,[7],1.0,[Giant's Belt]
0,KR_4291707834,platinum,5,4,4,Ashe,[9],1.0,[Sparring Gloves]
0,KR_4291707834,platinum,5,4,4,ChoGath,[6],1.0,[Negatron Cloak]
0,KR_4291707834,platinum,5,4,4,Ekko,[1],1.0,[B.F. Sword]


In [69]:
# 아이템 단위로 explode
df_champ_item_exploded2 = df_champ_item.explode('item_names')

# 챔피언별 아이템 사용 횟수 집계
champ_item_stats = (
    df_champ_item_exploded2
    .groupby(['name', 'item_names'])
    .size()
    .reset_index(name='count')
    .sort_values(['name', 'count'], ascending=[True, False])
)

# 챔피언별 Top 3 아이템 추출
champ_item_top3 = (
    champ_item_stats
    .groupby('name')
    .head(3)
    .reset_index(drop=True)
)

display(champ_item_top3.head(10))

,name,item_names,count
0,Ahri,Morellonomicon,3609
1,Ahri,Chalice of Favor,2656
2,Ahri,Thief's Gloves,1748
3,Annie,Zz'Rot Portal,2525
4,Annie,Quicksilver,2005
5,Annie,Bramble Vest,1555
6,Ashe,Spear of Shojin,11062
7,Ashe,Guinsoo's Rageblade,6127
8,Ashe,Statikk Shiv,6065
9,AurelionSol,Seraph's Embrace,2507


In [70]:
# 챔피언별 Top 3 아이템을 1행으로 합치기
champ_item_pivot = champ_item_top3.groupby('name')['item_names'].apply(
    lambda x: list(x)
).reset_index()

# item_1, item_2, item_3 컬럼으로 분리
champ_item_pivot[['item_1', 'item_2', 'item_3']] = pd.DataFrame(
    champ_item_pivot['item_names'].tolist(), index=champ_item_pivot.index
)

champ_item_pivot = champ_item_pivot.drop(columns='item_names')
display(champ_item_pivot.head(5))

,name,item_1,item_2,item_3
0,Ahri,Morellonomicon,Chalice of Favor,Thief's Gloves
1,Annie,Zz'Rot Portal,Quicksilver,Bramble Vest
2,Ashe,Spear of Shojin,Guinsoo's Rageblade,Statikk Shiv
3,AurelionSol,Seraph's Embrace,Morellonomicon,Demolitionist's Charge
4,Blitzcrank,Zephyr,Chalice of Favor,Rebel Medal


In [ ]:
# champion_item_top3.csv 저장
champ_item_pivot.to_csv('champion_item_top3.csv', index=False, encoding='utf-8-sig')

##### **✅ champion_item_top3.csv 파일의 컬럼 명세서**

| 컬럼명 | 타입 | 설명 | 예시 |
|---|---|---|---|
| name | str | 챔피언 이름 | Ahri |
| item_1 | str | 가장 많이 사용한 아이템 | Morellonomicon |
| item_2 | str | 두 번째로 많이 사용한 아이템 | Chalice of Favor |
| item_3 | str | 세 번째로 많이 사용한 아이템 | Thief's Gloves |

> 태블로에서 티어, 유저 순위, 챔피언 코스트 필터 적용 예정

---
#### champion_stats.csv 파일 제작

In [ ]:
# 현재 데이터셋에서 한 게임당 몇 명의 플레이어가 배정되어 있는지 확인
game_user_count = df_champion.groupby('game_id')['user_id'].nunique()
print(game_user_count.value_counts())

user_id
8    49339
7      209
1       11
6        3
Name: count, dtype: int64


In [81]:
# 전체 유저-게임 수 (gameid + user_id 기준 중복 제거)
total_user_games = len(df_champ_exploded[['game_id', 'user_id']].drop_duplicates())

# 챔피언별 통계 집계
champ_stats = df_champ_exploded.groupby(['name', 'cost']).agg(
    pick_count=('name', 'count'),
    avg_rank=('ranked', 'mean'),
    win_rate=('ranked_1', 'mean'),
    top4_rate=('top4_flag', 'mean')
).reset_index()

# 픽률 = pick_count(챔피언 단위) / total_user_games(유저-게임 단위)
champ_stats['pick_rate'] = champ_stats['pick_count'] / total_user_games

display(champ_stats.head(10))

,name,cost,pick_count,avg_rank,win_rate,top4_rate,pick_rate
0,ahri,2,36489,4.578256,0.131053,0.485407,0.092096
1,annie,2,52599,4.446111,0.134375,0.509991,0.132757
2,ashe,3,89109,4.389781,0.132085,0.517883,0.224907
3,aurelionsol,5,23117,3.538565,0.199723,0.688671,0.058346
4,blitzcrank,2,120040,4.412537,0.118386,0.517011,0.302975
5,caitlyn,1,21972,4.517704,0.122565,0.501729,0.055456
6,chogath,4,94857,4.252095,0.126306,0.547782,0.239415
7,darius,2,33400,4.687575,0.126677,0.458054,0.084300
8,ekko,5,45233,3.882365,0.167378,0.616629,0.114166
9,ezreal,3,105788,4.380932,0.118038,0.522488,0.267004


In [ ]:
# champion_stats.csv 저장 -> 티어 정보가 없는 전체 유저 기준의 데이터
champ_stats.to_csv('champ_stats.csv', index=False, encoding='utf-8-sig')

##### **✅ champion_stats.csv** 데이터 명세서

| 컬럼명 | 타입 | 설명 | 예시 |
|---|---|---|---|
| name | str | 챔피언 이름 | ahri |
| cost | int | 챔피언 코스트 | 2 |
| pick_count | int | 챔피언 등장 횟수 (챔피언 단위) | 36489 |
| avg_rank | float | 평균 등수 | 4.578 |
| win_rate | float | 승률 (1위 비율) | 0.131 |
| top4_rate | float | 순방율 (top4 비율) | 0.485 |
| pick_rate | float | 픽률 (pick_count / 전체 유저-게임 수) | 0.092 |


> 픽률이 1을 초과할 수 있음 (한 게임에서 여러 유저가 같은 챔피언을 선택 가능하기 때문)

> 태블로에서 티어, 유저 순위, 챔피언 코스트 필터 적용 예정

---
#### combination_stats.csv 제작

In [ ]:
# 조합별 통계 집계
# 픽률의 분모 - 동일한 게임+유저에 해당하는 행은 제거
total_user_games_combo = len(df_combination[['gameid', 'user_id']].drop_duplicates())

combo_stats = df_combo_active_exploded.groupby('combo_list_active').agg(
    pick_count=('combo_list_active', 'count'),
    avg_rank=('ranked', 'mean'),
    win_rate=('ranked_1', 'mean'),
    top4_rate=('top4_flag', 'mean')
).reset_index()

# 픽률 계산 = 전체 행에서 해당 조합이 선택된 행의 개수 / 전체 유저-게임 수
combo_stats['pick_rate'] = combo_stats['pick_count'] / total_user_games_combo

display(combo_stats.head(10))

,combo_list_active,pick_count,avg_rank,win_rate,top4_rate,pick_rate
0,Blademaster,103932,4.424624,0.127901,0.513547,0.262296
1,Blaster,115469,4.261672,0.129273,0.545081,0.291413
2,Brawler,101015,4.334059,0.122229,0.532792,0.254935
3,Celestial,137136,4.287058,0.142530,0.538247,0.346094
4,Chrono,200384,4.244925,0.138374,0.548607,0.505715
5,Cybernetic,41664,4.466446,0.123704,0.506384,0.105149
6,DarkStar,48084,4.311268,0.145059,0.530779,0.121351
7,Demolitionist,45954,4.091657,0.170497,0.571093,0.115975
8,Infiltrator,67183,4.302904,0.144039,0.537487,0.169552
9,ManaReaver,113311,4.167292,0.149165,0.560546,0.285966


In [ ]:
# combination_stats.csv 저장 -> 티어 정보가 없는 전체 유저 기준의 데이터
combo_stats.to_csv('combination_stats.csv', index=False, encoding='utf-8-sig')

##### **✅ combination_stats.csv 데이터명세서**

| 컬럼명 | 타입 | 설명 | 예시 |
|---|---|---|---|
| combo_list_active | str | 활성화된 조합 이름 | Chrono |
| pick_count | int | 해당 조합이 활성화된 횟수 | 200384 |
| avg_rank | float | 평균 등수 | 4.245 |
| win_rate | float | 승률 (1위 비율) | 0.138 |
| top4_rate | float | 순방율 (top4 비율) | 0.549 |
| pick_rate | float | 픽률 (pick_count / 전체 유저-게임 수) | 0.506 |

> 픽률 분모: 전체 유저-게임 수 (gameid + user_id 기준 중복 제거)
> 활성화된 조합만 포함 (active_synergies 기준)
> 태블로에서 티어, 유저 순위 필터 적용 예정

---
### champion, champion+item 테이블에 user_tier 붙이기
- 티어별로 필터를 걸 때 활용 가능

(1) champion_stats_by_tier.csv 제작

In [89]:
# 티어별 전체 유저-게임 수
total_user_games_by_tier = df_champ_exploded.groupby('user_tier')[['game_id', 'user_id']].apply(
    lambda x: len(x.drop_duplicates())
).reset_index(name='total_user_games')

# 챔피언별 통계 집계 (티어별)
champ_stats_by_tier = df_champ_exploded.groupby(['name', 'cost', 'user_tier']).agg(
    pick_count=('name', 'count'),
    avg_rank=('ranked', 'mean'),
    win_rate=('ranked_1', 'mean'),
    top4_rate=('top4_flag', 'mean')
).reset_index()

# 티어별 픽률 계산
champ_stats_by_tier = champ_stats_by_tier.merge(total_user_games_by_tier, on='user_tier')
champ_stats_by_tier['pick_rate'] = champ_stats_by_tier['pick_count'] / champ_stats_by_tier['total_user_games']
champ_stats_by_tier = champ_stats_by_tier.drop(columns='total_user_games')

display(champ_stats_by_tier.head(5))

,name,cost,user_tier,pick_count,avg_rank,win_rate,top4_rate,pick_rate
0,ahri,2,challenger,8499,4.515355,0.141428,0.495941,0.106344
1,ahri,2,diamond,5124,4.623341,0.119048,0.480874,0.064489
2,ahri,2,grand_master,7167,4.650481,0.135901,0.467281,0.089640
3,ahri,2,master,7832,4.551583,0.138151,0.490552,0.097986
4,ahri,2,platinum,7867,4.577603,0.116182,0.488369,0.102242


In [90]:
# champion_stats_by_tier.csv 저장 -> 유저의 티어 데이터가 포함된 데이터
champ_stats_by_tier.to_csv('champion_stats_by_tier.csv', index=False, encoding='utf-8-sig')

##### **✅ champion_stats_by_tier.csv 데이터 명세서**

| 컬럼명 | 타입 | 설명 | 예시 |
|---|---|---|---|
| name | str | 챔피언 이름 | ahri |
| cost | int | 챔피언 코스트 | 2 |
| user_tier | str | 유저 티어 | challenger |
| pick_count | int | 챔피언 등장 횟수 | 8499 |
| avg_rank | float | 평균 등수 | 4.515 |
| win_rate | float | 승률 (1위 비율) | 0.141 |
| top4_rate | float | 순방율 (top4 비율) | 0.496 |
| pick_rate | float | 픽률 (pick_count / 티어별 전체 유저-게임 수) | 0.106 |

> 태블로에서 티어, 유저 순위, 챔피언 코스트 필터 적용 예정

(2) combunation_stats_by_tier.csv 제작

In [91]:
# 티어별 전체 유저-게임 수
total_user_games_combo_by_tier = df_combination.groupby('user_tier')[['gameid', 'user_id']].apply(
    lambda x: len(x.drop_duplicates())
).reset_index(name='total_user_games')

# 조합별 통계 집계 (티어별)
combo_stats_by_tier = df_combo_active_exploded.groupby(['combo_list_active', 'user_tier']).agg(
    pick_count=('combo_list_active', 'count'),
    avg_rank=('ranked', 'mean'),
    win_rate=('ranked_1', 'mean'),
    top4_rate=('top4_flag', 'mean')
).reset_index()

# 티어별 픽률 계산
combo_stats_by_tier = combo_stats_by_tier.merge(total_user_games_combo_by_tier, on='user_tier')
combo_stats_by_tier['pick_rate'] = combo_stats_by_tier['pick_count'] / combo_stats_by_tier['total_user_games']
combo_stats_by_tier = combo_stats_by_tier.drop(columns='total_user_games')

display(combo_stats_by_tier.head(5))

,combo_list_active,user_tier,pick_count,avg_rank,win_rate,top4_rate,pick_rate
0,Blademaster,challenger,20926,4.443229,0.125538,0.509271,0.261820
1,Blademaster,diamond,22950,4.364488,0.132244,0.524793,0.288832
2,Blademaster,grand_master,21598,4.383415,0.128947,0.522224,0.270110
3,Blademaster,master,20253,4.460623,0.126055,0.507283,0.253384
4,Blademaster,platinum,18205,4.487888,0.125954,0.500961,0.236533


In [ ]:
# combination_stats_by_tier.csv 저장 -> 유저의 티어 데이터가 포함된 데이터
combo_stats_by_tier.to_csv('combination_stats_by_tier.csv', index=False, encoding='utf-8-sig')

##### **✅ combination_stats_by_tier.csv 데이터명세서**

| 컬럼명 | 타입 | 설명 | 예시 |
|---|---|---|---|
| combo_list_active | str | 활성화된 조합 이름 | Blademaster |
| user_tier | str | 유저 티어 | challenger |
| pick_count | int | 해당 조합이 활성화된 횟수 | 20926 |
| avg_rank | float | 평균 등수 | 4.443 |
| win_rate | float | 승률 (1위 비율) | 0.126 |
| top4_rate | float | 순방율 (top4 비율) | 0.509 |
| pick_rate | float | 픽률 (pick_count / 티어별 전체 유저-게임 수) | 0.262 |

> 활성화된 조합만 포함 (active_synergies 기준)
> 태블로에서 티어, 유저 순위 필터 적용 예정